In [5]:
# =====================================================================
#  CELLULE UNIQUE — ne jamais exécuter de morceau séparé de cette cellule
# =====================================================================
%matplotlib widget
import ipywidgets as widgets
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb
import math

# ---------- paramètres & constantes ----------
params = dict(zoom=1.0, orb_color='#FF0000', orb_scale=1.0, angle_step=15.0)
BOND = 1.2          # longueur de liaison imposée (unités du graphe)

# ============ Van-Catledge, J. Org. Chem. 1980, 45, 4801 ==============
# αX = α + hX·β ;  βXY = kXY·β
# Type 1 = 1 électron π ; type 2 = paire libre (2 électrons π)
# el : (n_π, h, valence π, couleur)
ATOMTYPES = {
    "C":  (1,  0.00, 3, '#333333'),
    "B":  (1, -0.45, 3, '#d9a441'),
    "N":  (1,  0.51, 2, '#3050c8'),   # N1 pyridinique : lone pair dans le plan
    "N:": (2,  1.37, 3, '#3050c8'),   # N2 pyrrole / amine : lone pair π
    "O":  (1,  0.97, 1, '#c83030'),   # O1 carbonyle
    "O:": (2,  2.09, 2, '#c83030'),   # O2 éther / furane
    "F":  (2,  2.71, 0, '#66c830'),   # substituant, 0 voisin π
    "Si": (1,  0.00, 3, '#a08060'),   # silabenzène
    "P":  (1,  0.19, 2, '#c88030'),   # P1 phospha-pyridine
    "P:": (2,  0.75, 3, '#c88030'),   # P2
    "S":  (1,  0.46, 1, '#c8c030'),   # S1 thiocarbonyle
    "S:": (2,  1.11, 2, '#c8c030'),   # S2 thioéther / thiophène
    "Cl": (2,  1.48, 0, '#30a060'),   # substituant, 0 voisin π
}

# ---- Table II : demi-matrice kXY (ordre des lignes = ordre d'ATOMTYPES) ----
KHALF = [
    [1.00],                                                         # C
    [0.73, 0.87],                                                   # B
    [1.02, 0.66, 1.09],                                             # N
    [0.89, 0.53, 0.99, 0.98],                                       # N:
    [1.06, 0.60, 1.14, 1.13, 1.26],                                 # O
    [0.66, 0.35, 0.80, 0.89, 1.02, 0.95],                           # O:
    [0.52, 0.26, 0.65, 0.77, 0.92, 0.94, 1.04],                     # F
    [0.75, 0.57, 0.72, 0.43, 0.65, 0.24, 0.17, 0.64],               # Si
    [0.77, 0.53, 0.78, 0.55, 0.75, 0.31, 0.21, 0.62, 0.63],         # P
    [0.76, 0.54, 0.81, 0.64, 0.82, 0.39, 0.22, 0.52, 0.58, 0.63],   # P:
    [0.81, 0.51, 0.83, 0.68, 0.84, 0.43, 0.28, 0.61, 0.65, 0.65, 0.68],          # S
    [0.69, 0.44, 0.78, 0.73, 0.85, 0.54, 0.32, 0.40, 0.48, 0.60, 0.58, 0.63],    # S:
    [0.62, 0.41, 0.77, 0.80, 0.88, 0.70, 0.51, 0.34, 0.35, 0.55, 0.52, 0.59, 0.68],  # Cl
]

# --- contrôles de cohérence : KHALF doit matcher ATOMTYPES ---
assert len(KHALF) == len(ATOMTYPES), "KHALF ne correspond pas à ATOMTYPES"
assert all(len(row) == i + 1 for i, row in enumerate(KHALF)), "KHALF mal formée"

# ---------- Défauts (atome/paire inconnu → neutre) ----------
K_DEFAULT       = 0.7
NPI_DEFAULT     = 0
H_DEFAULT       = 0.0
VAL_DEFAULT     = 3
COULEUR_DEFAULT = '#999999'

# ---------- Tables dérivées : une seule source de vérité ----------
NPI_TABLE     = {n: t[0] for n, t in ATOMTYPES.items()}
H_TABLE       = {n: t[1] for n, t in ATOMTYPES.items()}
VAL_TABLE     = {n: t[2] for n, t in ATOMTYPES.items()}
COULEUR_TABLE = {n: t[3] for n, t in ATOMTYPES.items()}

# ---------- kXY : dict plein symétrique construit depuis KHALF ----------
K_TABLE = {}
_names = list(ATOMTYPES)
for i, row in enumerate(KHALF):
    for j, k in enumerate(row):
        K_TABLE[frozenset((_names[i], _names[j]))] = k

def k_xy(el1, el2):
    """k de la paire (el1, el2), symétrique. K_DEFAULT si paire inconnue."""
    return K_TABLE.get(frozenset((el1, el2)), K_DEFAULT)

def h_x(el):
    """h du site el (αX = α + hX·β). H_DEFAULT si inconnu."""
    return H_TABLE.get(el, H_DEFAULT)

k_default = k_xy   # alias historique

# ---------- style CSS ----------
display(HTML("""
<style>
.mybtn button {
    height: 28px;
    border-radius: 10px !important; border: 1px solid #9a9a9a !important;
    background: linear-gradient(180deg,#fdfdfd 0%,#d8d8d8 45%,#b8b8b8 100%) !important;
    box-shadow: 0 2px 3px rgba(0,0,0,.35), inset 0 1px 0 rgba(255,255,255,.8) !important;
    font-weight: 600 !important; color: #222 !important;
}
.mybtn button:hover { background: linear-gradient(180deg,#fff 0%,#e6e6e6 45%,#ccc 100%) !important; }
.mybtn button:active { box-shadow: inset 0 2px 4px rgba(0,0,0,.4) !important; }
</style>"""))

# ---------- données moléculaires ----------
alpha, beta = 0.0, -1.0          # E = alpha + m*beta ; m>0 = liante

class Mol:
    def __init__(self):
        self.pos = [(0.0, 0.0)]; self.el = ['C']
        self.npi = [NPI_TABLE['C']]; self.val = [VAL_TABLE['C']]
        self.h = [H_TABLE['C']]
        self.q = 0        # charge globale en e⁻ (+1 = cation, −1 = anion)
        self.adj = [set()]
        self.kov = {}                                 # overrides {(min,max): k}
        self._cache = None

    @property
    def nat(self):
        return len(self.pos)

    @property
    def npi0(self):
        """Électrons π de chaque atome neutre (référence pour les charges)."""
        return np.array([NPI_TABLE.get(t, NPI_DEFAULT) for t in self.el])

    @property
    def charges(self):
        P = self.density()
        n0 = self.npi0
        return [n0[i] - P[i, i] for i in range(self.nat)]

    # ---------- snapshots ----------
    def snapshot(self):
        return (list(self.pos), list(self.el), list(self.npi), list(self.val),
                list(self.h), dict(self.kov), [set(s) for s in self.adj],self.q)

    def restore(self, s):
        (self.pos, self.el, self.npi, self.val, self.h, self.kov, self.adj,self.q) = s
        self._cache = None

    # ---------- réglages élément / h / k ----------
    def set_element(self, i, el):
        self.el[i] = el
        self.h[i]   = H_TABLE.get(el, H_DEFAULT)
        self.npi[i] = NPI_TABLE.get(el, NPI_DEFAULT)
        self.val[i] = VAL_TABLE.get(el, VAL_DEFAULT)
        self._cache = None

    def set_h(self, i, h):
        self.h[i] = h
        self._cache = None

    def set_k(self, i, j, k):
        key = (min(i, j), max(i, j))
        default = k_xy(self.el[i], self.el[j])
        if abs(k - default) < 1e-9:
            self.kov.pop(key, None)
        else:
            self.kov[key] = k
        self._cache = None

    def k_of(self, i, j):
        key = (min(i, j), max(i, j))
        if key in self.kov: return self.kov[key]
        return k_xy(self.el[i], self.el[j])

    # ---------- modifications ----------
    def _new_carbon(self, x, y):
        self.pos.append((x, y)); self.el.append('C')
        self.npi.append(NPI_TABLE['C']); self.val.append(VAL_TABLE['C'])
        self.h.append(H_TABLE['C'])
        self.adj.append(set())
        self._cache = None
        return len(self.pos) - 1

    def add_atom(self, x, y, i_src):
        if len(self.adj[i_src]) >= self.val[i_src]: return None
        for (px, py) in self.pos:
            if (px - x)**2 + (py - y)**2 < 0.3**2: return None
        j = self._new_carbon(x, y)
        self.adj[i_src].add(j); self.adj[j].add(i_src)
        self._cache = None
        return j

    def delete(self, i):
        if len(self.adj[i]) > 1: return False
        for j in self.adj[i]: self.adj[j].discard(i)
        for L in (self.pos, self.el, self.npi, self.val, self.h, self.adj):
            L.pop(i)
        new = {}
        for (a, b), k in self.kov.items():
            a2 = a - 1 if a > i else a
            b2 = b - 1 if b > i else b
            if a2 == b2: continue
            if a2 >= 0 and b2 >= 0:
                new[(min(a2, b2), max(a2, b2))] = k
        self.kov = new
        self._cache = None
        return True

    def add_bond_or_atom(self, x, y, i_src):
        for j, (px, py) in enumerate(self.pos):
            if j != i_src and (px - x)**2 + (py - y)**2 < 0.15**2:
                if j in self.adj[i_src] or len(self.adj[i_src]) >= self.val[i_src]:
                    return None
                self.adj[i_src].add(j); self.adj[j].add(i_src)
                self._cache = None
                return ('bond', j)
        if len(self.adj[i_src]) >= self.val[i_src]: return None
        j = self._new_carbon(x, y)
        self.adj[i_src].add(j); self.adj[j].add(i_src)
        self._cache = None
        return ('atom', j)

    def remove_bond(self, i, j):
        if j not in self.adj[i]: return False
        self.adj[i].discard(j); self.adj[j].discard(i)
        self.kov.pop((min(i, j), max(i, j)), None)
        self._cache = None
        return True

    # ---------- Hückel ----------
    def huckel(self):
        n = len(self.pos)
        H = np.zeros((n, n))
        for i in range(n):
            H[i, i] = alpha + self.h[i] * beta
            for j in self.adj[i]:
                if j > i:
                    H[i, j] = H[j, i] = self.k_of(i, j) * beta
        if n == 1: return np.array([H[0, 0]]), np.array([[1.0]])
        return np.linalg.eigh(H)

    def remplissage(self):
        if self._cache is None:
            e, c = self.huckel()
            m = (e - alpha) / beta
            order = np.argsort(-m)          # m décroissant : liante d'abord
            e = e[order]; c = c[:, order]
            ne = int(sum(self.npi))- self.q  # charge globale
            occ = np.zeros(len(e), dtype=int)
            for kk in range(len(e)):
                take = min(2, max(0, ne))
                occ[kk] = take; ne -= take
            self._cache = (e, c, occ)
        return self._cache

    def density(self):
        """Matrice densité P = Σ_occ n_k · c_k c_kᵀ (nat × nat)."""
        e, c, occ = self.remplissage()
        P = np.zeros((self.nat, self.nat))
        for k in range(len(e)):
            if occ[k] > 0:
                P += occ[k] * np.outer(c[:, k], c[:, k])
        return P


mol = Mol()
history = [mol.snapshot()]; future = []

# ---------- calculs ----------
def darken(color, f=0.90):
    r, g, b = to_rgb(color)
    return (r*f, g*f, b*f)

def bond_orders():
    e, c, occ = mol.remplissage()
    P = np.zeros((len(mol.pos), len(mol.pos)))
    for k in range(len(occ)):
        if occ[k]:
            P += occ[k] * np.outer(c[:, k], c[:, k])
    return P

def h_angles(i):
    x, y = mol.pos[i]
    occ = [math.degrees(math.atan2(mol.pos[j][1]-y, mol.pos[j][0]-x))
           for j in mol.adj[i]]
    nH = mol.val[i] - len(occ)
    if nH == 0: return []
    step = 360 / mol.val[i]

    def ang_dist(a, b):
        return abs(((a - b + 180) % 360) - 180)

    if not occ:
        return [90 + step*k for k in range(nH)]
    best, best_mis = None, 1e9
    for a0 in range(0, 360, 5):
        slots = [a0 + step*k for k in range(mol.val[i])]
        mis = sum(min(ang_dist(o, s) for s in slots) for o in occ)
        if mis < best_mis: best_mis, best = mis, slots
    free = sorted(best, key=lambda s: -min(ang_dist(s, o) for o in occ))
    return free[:nH]

def dist_point_segment(px, py, x1, y1, x2, y2):
    """distance du point au segment [A,B]"""
    vx, vy = x2 - x1, y2 - y1
    L2 = vx*vx + vy*vy
    if L2 == 0: return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1)*vx + (py - y1)*vy) / L2))
    return math.hypot(px - (x1 + t*vx), py - (y1 + t*vy))

def hit_bond(x, y, tol=0.12):
    """(i,j) si le point touche une liaison, sinon None"""
    for i in range(len(mol.pos)):
        for j in mol.adj[i]:
            if j > i:
                x1, y1 = mol.pos[i]; x2, y2 = mol.pos[j]
                if dist_point_segment(x, y, x1, y1, x2, y2) < tol:
                    return (i, j)
    return None

def print_MOs():
    """Tableau ASCII des OM."""
    e, c, occ = mol.remplissage()
    n = len(mol.pos); nmo = len(e)
    w = 11
    sep = '+' + ('-' * (w + 2)) * (nmo + 1) + '+'

    def row(label, cells):
        s = f'| {label:<{w}} |'
        for v in cells:
            s += f' {v:>{w}} |'
        return s

    print(sep)
    print(row('E (α + mβ)', [f'm = {(ev - alpha)/beta:+.2f}' for ev in e]))
    print(row('occ (e⁻)',  [str(o) for o in occ]))
    print(sep)
    for i in range(n):
        print(row(f'{i+1} {mol.el[i]}',
                  [f'{c[i, k]:+.3f}' for k in range(nmo)]))
    print(sep)
    P = bond_orders()
    print('Charges π qi :',
          ', '.join(f'{mol.el[i]}{i+1}: {P[i, i]:.3f}' for i in range(n)))

def print_density():
    """Matrice densité + charges π."""
    P = mol.density()
    n = P.shape[0]
    largeur = 8
    entete = ' ' * largeur + ''.join(f'{j+1:>{largeur}d}' for j in range(n))
    print('Matrice densité P (charges : diagonale)')
    print(entete)
    print('-' * len(entete))
    for i in range(n):
        ligne = f'{i+1:<{largeur-1}d} '
        for j in range(n):
            ligne += f'{P[i,j]:>{largeur}.3f}'
        print(ligne)
    print()
    print(f'Charges π par atome : (charge totale={sum(mol.charges):.3f} e-)')
    print('  ' + '  '.join(
        f'{i+1}: {q:+.3f}' for i, q in enumerate(mol.charges)))
    print('  ')
## ---------- rapport Hückel dans un widget Output ----------
def huckel_report(_=None):
    with out:
        out.clear_output()
        print_MOs()
        print()
        print('=' * 60)
        print()
        print_density()

# ---------- figure (créée UNE fois) ----------
fig = plt.figure(layout='constrained', figsize=(6.0, 4.0))
gs  = fig.add_gridspec(1, 2, width_ratios=[6, 1.5])
axMol = fig.add_subplot(gs[0]); axMO = fig.add_subplot(gs[1])
axMol.set_xlim(-6, 6); axMol.set_ylim(-4, 4)
axMol.set_aspect('equal'); axMol.axis('off')
axMO.set_xticks([]); axMO.set_title("OM", fontsize=9)

selected_mo = None
drag = None


# ---------- état global ----------
reorder_mode = False
new_order = []          # liste des indices cliqués, dans l'ordre
show_numbers = False


# ----------  REORDONNER LES ATOMES .. ----------
def enter_reorder():
    global reorder_mode, new_order
    if not reorder_mode:
        reorder_mode, new_order = True, []
        btn_reorder.button_style = 'success'
        btn_reorder.description = '✓'

def exit_reorder():
    global reorder_mode, new_order
    reorder_mode, new_order = False, []
    btn_reorder.button_style = ''
    btn_reorder.description = '⇅'

def apply_reorder():
    """Applique new_order ; les non-cliqués gardent leur ordre relatif, à la suite."""
    global selected_mo
    if not new_order:
        exit_reorder(); draw(); return
    rest  = [i for i in range(mol.nat) if i not in new_order]
    order = new_order + rest
    inv = [0]*mol.nat
    for new, old in enumerate(order):
        inv[old] = new
    mol.pos = [mol.pos[i] for i in order]
    mol.el  = [mol.el[i]  for i in order]
    mol.npi = [mol.npi[i] for i in order]
    mol.val = [mol.val[i] for i in order]
    mol.h   = [mol.h[i]   for i in order]
    mol.adj = [{inv[j] for j in mol.adj[i]} for i in order]      # ← set, pas list
    mol.kov = {(min(inv[a], inv[b]), max(inv[a], inv[b])): k
               for (a, b), k in mol.kov.items()}
    selected_mo = None
    exit_reorder()
    push_history()
    draw()

def reorder_atoms(btn=None):       # bouton ⇅ : entre, ou valide si déjà en mode
    if reorder_mode:
        apply_reorder()
    else:
        enter_reorder()
        draw()                              # ← ICI : estompage visible avant le 1er clic

def handle_reorder_click(x, y):
    for i, (ax_, ay_) in enumerate(mol.pos):
        if (x - ax_)**2 + (y - ay_)**2 < 0.25**2:
            if i not in new_order:
                new_order.append(i)
                if mol.nat > 1 and len(new_order) >= mol.nat - 1:
                    apply_reorder()          # ← sort du mode, ne peut plus réarmer
                    return True
                draw()                      # ← ICI : suite du mode (gris + numéro vert)
            else:
                draw()
            return True
    return False
# ---------- FIN reord---------- 


# ---------- dessin ----------
def draw_H():
    for i, (x, y) in enumerate(mol.pos):
        for a in h_angles(i):
            ca, sa = math.cos(math.radians(a)), math.sin(math.radians(a))
            hx, hy = x + 0.55*ca, y + 0.55*sa
            axMol.plot([x + 0.20*ca, hx], [y + 0.20*sa, hy],
                       color='0.55', lw=0.8, zorder=1)
            axMol.add_patch(plt.Circle((hx, hy), 0.11, color='0.75', zorder=3))
            axMol.text(hx, hy, "H", ha='center', va='center',
                       fontsize=8, color='black', zorder=4)

def draw_MO():
    axMO.clear(); axMO.set_xticks([])
    axMO.set_title("OM" + (f" ({mol.q:+d})" if mol.q else ""), fontsize=9)
    e, c, occ = mol.remplissage()
    m = (e - alpha) / beta
    axMO.set_xlim(-.6, .6)
    axMO.set_ylim(m.max() + 1.5, m.min() - 1.0)
    for kk, (mk, ok) in enumerate(zip(m, occ)):
        lw = 3.5 if kk == selected_mo else 3
        axMO.hlines(mk, -0.35, 0.35, color='black', lw=lw)
        axMO.text(.40, mk, rf'$\alpha{mk:+.2f}\beta$', fontsize=6, va='center')
        if ok >= 1:
            axMO.annotate('', xy=(-.10, mk-.25), xytext=(-.10, mk),
                          arrowprops=dict(arrowstyle='-|>', color='red', lw=1.2))
        if ok == 2:
            axMO.annotate('', xy=(-.02, mk), xytext=(-.02, mk-.25),
                          arrowprops=dict(arrowstyle='-|>', color='red', lw=1.2))
    fig.canvas.draw_idle()

def draw(_=None):
    axMol.clear(); axMol.axis('off')
    axMol.set_aspect('equal')
    axMol.set_xlim(-6 / params['zoom'], 6 / params['zoom'])
    axMol.set_ylim(-4 / params['zoom'], 4 / params['zoom'])

    # ==============================================================
    # 1) INDICES DE LIAISON
    #    P[i,j] = ordre de liaison (Hückel), utilisé pour les
    #    pointillés qui marquent les liaisons partielles.
    # ==============================================================
    P = bond_orders() if len(mol.pos) > 1 else np.zeros((1, 1))

    # ==============================================================
    # 2) HYDROGÈNES IMPLICITES
    #    dessinés autour des atomes (ne comptent pas dans adj).
    # ==============================================================
    draw_H()

    # ==============================================================
    # 3) LIAISONS
    #    - couleur orange si override manuel (mol.kov), noir sinon
    #    - pointillés si ordre de liaison > 0.35
    #      (densité des pointillés ~ p)
    #    - en mode réordonnancement : estompées si un atome
    #      de la liaison n'est pas encore classé
    # ==============================================================
    for i in range(len(mol.pos)):
        for j in list(mol.adj[i]):
            if j > i:                          # chaque liaison une seule fois
                x1, y1 = mol.pos[i]; x2, y2 = mol.pos[j]
                base_col = '#EE7733' if (i, j) in mol.kov else 'k'
                a_b = 1.0 if (not reorder_mode
                              or (i in new_order and j in new_order)) else 0.25
                axMol.plot([x1, x2], [y1, y2], '-', color=base_col,
                           lw=1.5, alpha=a_b, zorder=2)

                # pointillés de liaison partielle (2e trait, décalé)
                p = P[i, j]
                if p > 0.05:
                        dx, dy = x2 - x1, y2 - y1
                        L = np.hypot(dx, dy) or 1
                        ux, uy = dx / L, dy / L
                        ox, oy = -uy * 0.12, ux * 0.12

                        a, b = 0.18, 0.82
                        seg_on  = L * (0.030 + 0.050 * p)        # tiret : 3% → 8% de L
                        seg_off = L * (0.010 + 0.230 * (1 - p))   # trou  : 1% → 24% de L

                        t = a * L
                        while t < b * L:
                            t2 = min(t + seg_on, b * L)
                            axMol.plot([x1 + ux*t + ox, x1 + ux*t2 + ox],
                                       [y1 + uy*t + oy, y1 + uy*t2 + oy],
                                       '-', color=base_col, lw=1.3,
                                       solid_capstyle='butt', zorder=3)
                            t += seg_on + seg_off    # ==============================================================
    # 4) OM SÉLECTIONNÉ (si demandé)
    #    deux cercles par atome : un décalé (phase opposée, blanc),
    #    un centré (phase de c[i, mo]), rayon ~ |coefficient|.
    # ==============================================================
    if selected_mo is not None:
        e, c, _ = mol.remplissage()
        decalage = 0.1
        for i, (x, y) in enumerate(mol.pos):
            r = params['orb_scale'] * 0.8 * abs(c[i, selected_mo])
            if r <= 0.02: continue
            s = np.sign(c[i, selected_mo])
            front_col = params['orb_color'] if s > 0 else 'white'
            back_col  = 'white' if s > 0 else params['orb_color']
            axMol.add_patch(plt.Circle((x - decalage, y + decalage), r,
                             facecolor=back_col, alpha=0.75,
                             edgecolor=darken(params['orb_color']),
                             lw=1.2, zorder=1.0))
            axMol.add_patch(plt.Circle((x, y), r, facecolor=front_col,
                             alpha=0.75, edgecolor=darken(params['orb_color']),
                             lw=1.2, zorder=1.1))

    # ==============================================================
    # 5) ATOMES
    #    disque coloré (couleur de l'élément) + symbole, nb d'e- π,
    #    nb d'H implicites.
    #    En mode réordonnancement :
    #      - alpha 0.25 pour les non classés
    #      - numéro du nouvel ordre en vert au-dessus des classés
    # ==============================================================
    for i, (x, y) in enumerate(mol.pos):
        col = COULEUR_TABLE.get(mol.el[i], COULEUR_DEFAULT)
        a = 1.0 if (not reorder_mode or i in new_order) else 0.25
        axMol.add_patch(plt.Circle((x, y), 0.22, facecolor=col,
                          edgecolor='k', alpha=a, zorder=3))
#       nH = mol.val[i] - len(mol.adj[i])
#       axMol.text(x, y, f"{mol.el[i]}\n{mol.npi[i]}e" + (f" ({nH}H)" if nH else ""),
        axMol.text(x, y, f"{mol.el[i]}",
                   ha='center', va='center', fontsize=9,
                   color='white', zorder=4)
        # numéro du nouvel ordre (mode réordonnancement)
        if reorder_mode and i in new_order:
            axMol.text(x, y + 0.32, str(new_order.index(i) + 1),
                       ha='center', fontsize=9, fontweight='bold',
                       color='#00ff88', zorder=6)
    
    # ==============================================================
    # 6) LIAISON EN COURS DE CRÉATION (drag en cours)
    #    trait pointillé de l'atome source au curseur.
    # ==============================================================
    if drag:
        x1, y1 = mol.pos[drag[0]]; x2, y2 = drag[2]
        axMol.plot([x1, x2], [y1, y2], 'k--', lw=1, zorder=2)

    # ==============================================================
    # 7) numerotation 
    # ==============================================================
    if show_numbers:
        for i, (x, y) in enumerate(mol.pos):
            axMol.text(x, y + 0.28, str(i+1), ha='center', va='center',
                       fontsize=9, fontweight='bold',
                       color='#001188', zorder=16)

    # ==============================================================
    # 7) PANNEAU OM (diagramme d'énergies)
    # ==============================================================
    draw_MO()


# ---------- interactions souris ----------
def on_click(ev):
    global drag, selected_mo, reorder_mode
    # ----- clic dans le diagramme MO : sélection / désélection (toggle) -----
    if ev.inaxes == axMO:
        e, _, _ = mol.remplissage()
        if len(e):
            m = (e - alpha) / beta            # unités de l'axe, pas des énergies brutes
            k = int(np.argmin(np.abs(m - ev.ydata)))
            selected_mo = None if k == selected_mo else k
            draw()
        return

    # ----- en dehors de la zone molécule, ou autre bouton : rien -----
    if ev.inaxes != axMol or ev.button != 1:
        return

    # ----- mode réordonnancement : chaque clic ajoute un atome à la liste -----
    if reorder_mode:
        handle_reorder_click(ev.xdata, ev.ydata)
        return                                   # clic à côté : ignoré en mode reorder

    # ----- mode normal : clic sur un atome -> début de drag -----
    for i, (x, y) in enumerate(mol.pos):
        if (ev.xdata - x)**2 + (ev.ydata - y)**2 < 0.25**2:
            drag = [i, (ev.xdata, ev.ydata), (ev.xdata, ev.ydata)]
            return

    # ----- clic à vide : peut-être une liaison -----
    hit = hit_bond(ev.xdata, ev.ydata)
    if hit is not None:
        open_bond_editor(*hit)

def on_move(ev):
    if not drag or ev.inaxes != axMol: return
    x0, y0 = mol.pos[drag[0]]
    mx, my = ev.xdata, ev.ydata
    dx, dy = mx - x0, my - y0
    if np.hypot(dx, dy) < 0.15: return
    ang = np.degrees(np.arctan2(dy, dx))
    ang = round(ang / params['angle_step']) * params['angle_step']
    drag[2] = (x0 + BOND * np.cos(np.radians(ang)),
               y0 + BOND * np.sin(np.radians(ang)))
    best, bd = None, 0.55
    for j, (px, py) in enumerate(mol.pos):
        if j == drag[0] or j in mol.adj[drag[0]]: continue
        d = (px - mx)**2 + (py - my)**2
        if d < bd**2:
            best, bd = j, np.sqrt(d)
    if best is not None:
        drag[2] = mol.pos[best]
    draw()

def on_release(ev):
    global drag
    if not drag: return
    (x0, y0), (x1, y1) = drag[1], drag[2]
    moved = np.hypot(x1 - x0, y1 - y0)
    if moved < 0.15:
        open_atom_editor(drag[0])
    else:
        res = mol.add_bond_or_atom(x1, y1, drag[0])
        if res is not None:
            history.append(mol.snapshot()); future.clear()
    drag = None
    draw()

fig.canvas.mpl_connect('button_press_event', on_click)
fig.canvas.mpl_connect('motion_notify_event', on_move)
fig.canvas.mpl_connect('button_release_event', on_release)
fig.canvas.header_visible = False

# ---------- éditeurs (dans le panel Hückel) ----------
def push_history():
    history.append(mol.snapshot()); future.clear()

def close_editor(_=None):
    zone_huckel.children = [fronton_hk, huckel_bar]

def open_atom_editor(i):
    # grille de boutons étroits : 2 colonnes de ~70px
    btns = []
    for el, (npi, h, val, col) in ATOMTYPES.items():
        def cb(_, i=i, el=el, npi=npi, val=val):
            mol.set_element(i, el)   # met à jour el, h, npi, val en bloc
            push_history()
            close_editor(); draw()
        b = widgets.Button(description=f"{el} {npi}e", layout=widgets.Layout(width='70px'))
        b.add_class('mybtn'); b.style.button_color = col
        b.on_click(cb); btns.append(b)
    grid = widgets.GridBox(children=btns,
                           layout=widgets.Layout(grid_template_columns='70px 70px',
                                                  grid_gap='3px', width='auto'))

    # overrides k éventuels sur les liaisons de cet atome
    ov = [(j, mol.kov[(min(i, j), max(i, j))])
          for j in mol.adj[i] if (min(i, j), max(i, j)) in mol.kov]
    ov_html = widgets.HTML("<b>Overrides k :</b><br>" +
            ("<br>".join(f"→ {j+1} : {k:+.2f}" for j, k in ov) or "<i>aucun</i>"))

    def del_atom(_):
        if mol.delete(i):
            push_history()
        close_editor(); draw()
    bd = widgets.Button(description='delete', layout=widgets.Layout(width='143px'))
    bd.add_class('mybtn'); bd.on_click(del_atom)

    def set_h(change):
        mol.set_h(i, change['new'])
        mol.el[i] = 'X'
        draw()
    hfield = widgets.FloatText(value=mol.h[i], step=0.05,
                               description='h:', layout=widgets.Layout(width='143px'))
    hfield.observe(set_h, 'value')

    bc = widgets.Button(description='✕ Fermer', layout=widgets.Layout(width='143px'))
    bc.add_class('mybtn'); bc.on_click(close_editor)

    zone_huckel.children = [fronton_hk,
        widgets.VBox([widgets.HTML(f'<b>Atome {i+1} ?</b>'),
                      grid, hfield, ov_html, bd, bc],
                     layout=widgets.Layout(padding='4px'))]

def open_bond_editor(i, j):
    ij = (min(i, j), max(i, j))
    el1, el2 = mol.el[ij[0]], mol.el[ij[1]]
    kv = mol.kov.get(ij, k_xy(el1, el2))

    def set_k(change):
        mol.set_k(ij[0], ij[1], change['new'])
        push_history(); draw()
    kfield = widgets.FloatText(value=kv, step=0.05,
                               description='k:', layout=widgets.Layout(width='143px'))
    kfield.observe(set_k, 'value')

    def to_table(_):
        kfield.value = k_xy(mol.el[ij[0]], mol.el[ij[1]]); draw()
    def to_simple(_):
        kfield.value = 0.0; draw()
    def del_bond(_):
        if mol.remove_bond(ij[0], ij[1]):
            push_history()
        close_editor(); draw()

    bt = widgets.Button(description=f'Table ({k_xy(el1, el2):.2f})',
                        layout=widgets.Layout(width='143px'))
    bt.add_class('mybtn'); bt.on_click(to_table)
    bs = widgets.Button(description='Simple (k=0)', layout=widgets.Layout(width='143px'))
    bs.add_class('mybtn'); bs.on_click(to_simple)
    bb = widgets.Button(description='Supprimer', layout=widgets.Layout(width='143px'))
    bb.add_class('mybtn'); bb.on_click(del_bond)
    bc = widgets.Button(description='✕ Fermer', layout=widgets.Layout(width='143px'))
    bc.add_class('mybtn'); bc.on_click(close_editor)

    zone_huckel.children = [fronton_hk,
        widgets.VBox([widgets.HTML(f'<b>Liaison {ij[0]+1}—{ij[1]+1} '
                                   f'({el1}–{el2})</b>'),
                      kfield, bt, bs, bb, bc],
                     layout=widgets.Layout(padding='4px'))]

# ---------- boutons ----------
def mk(desc, cb=None, w='110px'):
    b = widgets.Button(description=desc, layout=widgets.Layout(width=w))
    b.add_class('mybtn')
    if cb: b.on_click(cb)
    return b

def undo(_):
    if len(history) > 1:
        future.append(history.pop()); mol.restore(history[-1]); draw()
        close_editor()
def redo(_):
    if future:
        s = future.pop(); history.append(s); mol.restore(s); draw()
        close_editor()   
def toggle_numbers(btn=None):
    global show_numbers
    show_numbers = not show_numbers
    draw()


def add_electron(_):
    mol.q -= 1                     # −1 charge = +1 électron π
    mol._cache = None
    push_history(); draw()
def remove_electron(_):
    mol.q += 1                     # +1 charge = −1 électron π
    mol._cache = None
    push_history(); draw()
def zi(_): params['zoom'] = min(params['zoom'] * 1.25, 8); draw()
def zo(_): params['zoom'] = max(params['zoom'] / 1.25, 0.5); draw()

dessin_bar = widgets.HBox([mk('⟲ Undo', undo), mk('⟳ Redo', redo),
                           mk('Zoom +', zi), mk('Zoom −', zo),
                           mk('⚙ Réglages'), mk('À propos')],
                          layout=widgets.Layout(width='100%',
                                                justify_content='flex-start'))

# ---------- panel Hückel : bouton Results + zone de sortie ----------
btn_reorder = mk('⇅', reorder_atoms, w='auto')
huckel_bar = widgets.GridBox(children=[
    mk('⟳', w='auto'),
    mk('i', toggle_numbers, w='auto'),
    btn_reorder,                     
    mk('hij', w='auto'),        # h_ij
    mk('qi', w='auto'),
    mk('− e⁻', remove_electron, w='auto'),
    mk('+ e⁻', add_electron, w='auto'),
    mk('Results', huckel_report, w='auto')],
    layout=widgets.Layout(grid_template_columns='1fr 1fr',
                          grid_gap='4px', width='100%'))

lewis_bar = widgets.GridBox(children=[mk('Generate all', w='auto'), mk('Create', w='auto'),
                                      mk('Results', w='auto'), mk('Erase 1', w='auto'),
                                      mk('Erase all', w='auto')],
                            layout=widgets.Layout(grid_template_columns='1fr 1fr',
                                                  grid_gap='4px', width='100%'))

def fronton(txt, bg):
    return widgets.HTML(f'<div style="background:{bg};text-align:center;'
                        f'color:black;font-weight:bold;font-size:15px;'
                        f'padding:2px;">{txt}</div>')

fronton_hk = fronton('Hückel', '#4499EE')

out = widgets.Output(layout=widgets.Layout(width='100%',
                                           border='2px solid #4499EE',
                                           padding='6px'))

zone_dessin = widgets.VBox([dessin_bar, fig.canvas],
                           layout=widgets.Layout(flex='1', width='auto',
                                                 min_width='0'))

zone_huckel = widgets.VBox([fronton('Hückel', '#4499EE'), huckel_bar],
                           layout=widgets.Layout(width='160px', background='#77CCEE',
                                                 border='2px solid #4499EE', padding='4px'))

zone_lewis  = widgets.VBox([fronton('Lewis', '#EE6633'), lewis_bar],
                          layout=widgets.Layout(width='160px', background='#FFCC66',
                                                border='2px solid #EE6633', padding='4px'))

main_row = widgets.HBox([zone_huckel, zone_dessin, zone_lewis],
                        layout=widgets.Layout(width='100%'))

dashboard = widgets.VBox([main_row, out],        # log sous les 3 panneaux
                         layout=widgets.Layout(width='100%'))

display(dashboard)      # ← dashboard, pas main_row !

draw()

from matplotlib._pylab_helpers import Gcf
Gcf.figs.pop(fig.number, None)
None

In [2]:
print(mk)  # dans une nouvelle cellule

<function mk at 0x149167ea45e0>
